In [80]:
import pandas as pd
import numpy as np

## Colunas que iremos trabalhar no dataset:<br>
**contador: identificador do registro<br>**
**TIPOOBITO: se é um óbito fetal ou não<br>**
**DTOBITO: data em que ocorreu o óbito<br>**
**DTNASC: data de nascimento<br>**
**IDADE: campo que mostra em minutos, horas, dias ou anos<br>**

In [81]:
# 1. Define a lista de colunas que serão importadas
colunas_desejadas = [
    'contador',
    'TIPOBITO',
    'DTOBITO',
    'DTNASC',
    'IDADE',
    'SEXO',
    'RACACOR',
    'ESTCIV',
    'ESC2010',
    'OCUP',
    'CODMUNRES',
    'CODMUNOCOR',
    'CAUSABAS',
    'ACIDTRAB'
]

# 2. Defina os tipos de dados das colunas
tipos_dados = {
    'TIPOBITO': str,
    'IDADE': str,       # Mantém como texto para não perder os zeros à esquerda da codificação do DATASUS
    'SEXO': str,
    'RACACOR': str,
    'ESTCIV': str,
    'ESC2010': str,
    'OCUP': str,
    'CODMUNRES': int,
    'CODMUNOCOR': str,
    'CAUSABAS': str     # Garante que códigos CID mistos não gerem alertas
}

# 3. Importa o arquivo CSV de forma otimizada
mortalidade = pd.read_csv(
    'Mortalidade_Geral_2026.csv',
    sep=';',
    usecols=colunas_desejadas,
    dtype=tipos_dados,
    parse_dates=['DTOBITO', 'DTNASC'],
    date_format='%d/%m/%Y'
)

# 4. Converte DTOBITO e DTNASC (formato: ddmmYYYY) para uma data real do Pandas (formato: YYYY-mm-dd)
mortalidade['DTOBITO'] = pd.to_datetime(mortalidade['DTOBITO'], format='%d%m%Y', errors='coerce')
mortalidade['DTNASC'] = pd.to_datetime(mortalidade['DTNASC'], format='%d%m%Y', errors='coerce')

In [82]:
mortalidade.head()

,contador,TIPOBITO,DTOBITO,DTNASC,IDADE,SEXO,RACACOR,ESTCIV,ESC2010,OCUP,CODMUNRES,CODMUNOCOR,CAUSABAS,ACIDTRAB
0,1,2,2026-04-03,1928-02-06,498,2,1,3,NaN,999992,310620,310620,I499,NaN
1,2,2,2026-03-11,1935-08-02,490,1,1,2,1,999993,430258,430258,C857,NaN
2,3,2,2026-01-28,1974-05-24,451,1,4,9,2,622020,230530,230530,I210,NaN
3,4,2,2026-02-07,1929-02-08,496,2,1,3,1,622020,230530,230530,R961,NaN
4,5,2,2026-02-19,1930-08-17,495,2,1,1,0,622020,230530,230530,R961,NaN


In [83]:
mortalidade.info()

<class 'pandas.DataFrame'>
RangeIndex: 506531 entries, 0 to 506530
Data columns (total 14 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   contador    506531 non-null  int64         
 1   TIPOBITO    506531 non-null  str           
 2   DTOBITO     506531 non-null  datetime64[us]
 3   DTNASC      505904 non-null  datetime64[us]
 4   IDADE       506531 non-null  str           
 5   SEXO        506531 non-null  str           
 6   RACACOR     500286 non-null  str           
 7   ESTCIV      486504 non-null  str           
 8   ESC2010     477991 non-null  str           
 9   OCUP        446342 non-null  str           
 10  CODMUNRES   506531 non-null  int64         
 11  CODMUNOCOR  506531 non-null  str           
 12  CAUSABAS    506531 non-null  str           
 13  ACIDTRAB    17986 non-null   float64       
dtypes: datetime64[us](2), float64(1), int64(2), str(9)
memory usage: 54.1 MB


In [84]:
municipios = pd.read_excel('MUNICIPIOS.xlsx')

In [85]:
municipios.head()

,CODMUNRES,MUNICIPIO,UF,POPULAÇÃO
0,355030,SÃO PAULO,SP,12200180.0
1,120030,FEIJÓ,AC,35035.0
2,251600,SOLÂNEA,PB,26777.0
3,260290,CABO DE SANTO AGOSTINHO,PE,203084.0
4,230210,BATURITÉ,CE,33335.0


In [86]:
municipios.info()

<class 'pandas.DataFrame'>
RangeIndex: 5580 entries, 0 to 5579
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CODMUNRES  5580 non-null   int64  
 1   MUNICIPIO  5580 non-null   str    
 2   UF         5580 non-null   str    
 3   POPULAÇÃO  5559 non-null   float64
dtypes: float64(1), int64(1), str(2)
memory usage: 174.5 KB


**Abaixo feito um merge com a tabela de municipios para poder analisar os dados de cada municipio**

In [87]:
mortalidade = mortalidade.merge(municipios, how='left')

In [88]:
mortalidade.head()

,contador,TIPOBITO,DTOBITO,DTNASC,IDADE,SEXO,RACACOR,ESTCIV,ESC2010,OCUP,CODMUNRES,CODMUNOCOR,CAUSABAS,ACIDTRAB,MUNICIPIO,UF,POPULAÇÃO
0,1,2,2026-04-03,1928-02-06,498,2,1,3,NaN,999992,310620,310620,I499,NaN,BELO HORIZONTE,MG,2392678.0
1,2,2,2026-03-11,1935-08-02,490,1,1,2,1,999993,430258,430258,C857,NaN,BOZANO,RS,2135.0
2,3,2,2026-01-28,1974-05-24,451,1,4,9,2,622020,230530,230530,I210,NaN,IBIAPINA,CE,23966.0
3,4,2,2026-02-07,1929-02-08,496,2,1,3,1,622020,230530,230530,R961,NaN,IBIAPINA,CE,23966.0
4,5,2,2026-02-19,1930-08-17,495,2,1,1,0,622020,230530,230530,R961,NaN,IBIAPINA,CE,23966.0


In [89]:
mortalidade.info()

<class 'pandas.DataFrame'>
RangeIndex: 506531 entries, 0 to 506530
Data columns (total 17 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   contador    506531 non-null  int64         
 1   TIPOBITO    506531 non-null  str           
 2   DTOBITO     506531 non-null  datetime64[us]
 3   DTNASC      505904 non-null  datetime64[us]
 4   IDADE       506531 non-null  str           
 5   SEXO        506531 non-null  str           
 6   RACACOR     500286 non-null  str           
 7   ESTCIV      486504 non-null  str           
 8   ESC2010     477991 non-null  str           
 9   OCUP        446342 non-null  str           
 10  CODMUNRES   506531 non-null  int64         
 11  CODMUNOCOR  506531 non-null  str           
 12  CAUSABAS    506531 non-null  str           
 13  ACIDTRAB    17986 non-null   float64       
 14  MUNICIPIO   506531 non-null  str           
 15  UF          506531 non-null  str           
 16  POPULAÇÃO   5

In [90]:
cid = pd.read_excel('CIDs_com_descricao.xlsx')

In [91]:
cid.head()

,CAUSABAS,DESCRICAO_CID
0,J449,Doença pulmonar obstrutiva crônica não especif...
1,W749,Afogamento e submersão não especificados - loc...
2,E149,Diabetes mellitus não especificado - sem compl...
3,C61,Neoplasia maligna da próstata
4,G309,Doença de Alzheimer não especificada


**feito um merge com a tabela de cids, para poder identificar as maiores causas de morte em 2026**

In [92]:
mortalidade = mortalidade.merge(cid, how='left', left_on='CAUSABAS', right_on='CAUSABAS')

In [93]:
mortalidade.head()

,contador,TIPOBITO,DTOBITO,DTNASC,IDADE,SEXO,RACACOR,ESTCIV,ESC2010,OCUP,CODMUNRES,CODMUNOCOR,CAUSABAS,ACIDTRAB,MUNICIPIO,UF,POPULAÇÃO,DESCRICAO_CID
0,1,2,2026-04-03,1928-02-06,498,2,1,3,NaN,999992,310620,310620,I499,NaN,BELO HORIZONTE,MG,2392678.0,Arritmia cardíaca não especificada
1,2,2,2026-03-11,1935-08-02,490,1,1,2,1,999993,430258,430258,C857,NaN,BOZANO,RS,2135.0,Outros tipos especificados de linfoma não-Hodgkin
2,3,2,2026-01-28,1974-05-24,451,1,4,9,2,622020,230530,230530,I210,NaN,IBIAPINA,CE,23966.0,Infarto agudo transmural da parede anterior do...
3,4,2,2026-02-07,1929-02-08,496,2,1,3,1,622020,230530,230530,R961,NaN,IBIAPINA,CE,23966.0,Morte que ocorre em menos de 24 horas após o i...
4,5,2,2026-02-19,1930-08-17,495,2,1,1,0,622020,230530,230530,R961,NaN,IBIAPINA,CE,23966.0,Morte que ocorre em menos de 24 horas após o i...


In [94]:
mortalidade.info()

<class 'pandas.DataFrame'>
RangeIndex: 506531 entries, 0 to 506530
Data columns (total 18 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   contador       506531 non-null  int64         
 1   TIPOBITO       506531 non-null  str           
 2   DTOBITO        506531 non-null  datetime64[us]
 3   DTNASC         505904 non-null  datetime64[us]
 4   IDADE          506531 non-null  str           
 5   SEXO           506531 non-null  str           
 6   RACACOR        500286 non-null  str           
 7   ESTCIV         486504 non-null  str           
 8   ESC2010        477991 non-null  str           
 9   OCUP           446342 non-null  str           
 10  CODMUNRES      506531 non-null  int64         
 11  CODMUNOCOR     506531 non-null  str           
 12  CAUSABAS       506531 non-null  str           
 13  ACIDTRAB       17986 non-null   float64       
 14  MUNICIPIO      506531 non-null  str           
 15  UF         

In [95]:
mortalidade['CAUSABAS'].isna().sum()

np.int64(0)

In [96]:
mortalidade['DESCRICAO_CID'].isna().sum()

np.int64(9044)

**cids que não foram localizados na tabela oficial**

In [97]:
mortalidade.loc[mortalidade['DESCRICAO_CID'].isna(), ['CAUSABAS', 'DESCRICAO_CID']]

,CAUSABAS,DESCRICAO_CID
107,L899,NaN
159,A090,NaN
200,C809,NaN
243,A090,NaN
301,J987,NaN
...,...,...
506294,I489,NaN
506353,C800,NaN
506385,C800,NaN
506389,C800,NaN


In [98]:
cids_faltantes = pd.DataFrame(cid.loc[cid['DESCRICAO_CID'].isna(), ['CAUSABAS', 'DESCRICAO_CID']])

In [99]:
cids_faltantes.to_csv('CIDs_faltantes.csv', index=False)

In [101]:
cid.loc[cid['CAUSABAS']=='A090', ['CAUSABAS', 'DESCRICAO_CID']]

,CAUSABAS,DESCRICAO_CID
381,A090,NaN


**10 maiores causas de morte em 2026:**

In [102]:
mortalidade['DESCRICAO_CID'].value_counts().head(10)

DESCRICAO_CID
Infarto agudo do miocárdio não especificado                                   27211
Pneumonia não especificada                                                    20220
Outras causas mal definidas e as não especificadas de mortalidade             17991
Infecção do trato urinário de localização não especificada                    11127
Hipertensão essencial (primária)                                              10751
Neoplasia maligna dos brônquios ou pulmões, não especificado                  10447
Acidente vascular cerebral, não especificado como hemorrágico ou isquêmico     9822
Doença de Alzheimer não especificada                                           9636
Septicemia não especificada                                                    7915
Neoplasia maligna da mama, não especificada                                    7155
Name: count, dtype: int64

**10 maiores municipios em quantidade de mortes em 2026:**

In [103]:
mortalidade['MUNICIPIO'].value_counts().head(10)

MUNICIPIO
SÃO PAULO         25266
RIO DE JANEIRO    19883
FORTALEZA          6520
SALVADOR           5714
BELO HORIZONTE     5630
BRASILIA           5168
CURITIBA           4709
RECIFE             4427
PORTO ALEGRE       4308
MANAUS             4167
Name: count, dtype: int64

In [104]:
acidentes = mortalidade[mortalidade['ACIDTRAB']== 1.0]

In [105]:
acidentes.head()

,contador,TIPOBITO,DTOBITO,DTNASC,IDADE,SEXO,RACACOR,ESTCIV,ESC2010,OCUP,CODMUNRES,CODMUNOCOR,CAUSABAS,ACIDTRAB,MUNICIPIO,UF,POPULAÇÃO,DESCRICAO_CID
117,118,2,2026-01-31,1986-08-13,439,1,4,2,2,622020,140023,140023,W208,1.0,CAROEBE,RR,10555.0,"Impacto causado por objeto lançado, projetado ..."
2653,2654,2,2026-01-24,1957-11-15,468,1,1,2,2,NaN,500410,500580,V849,1.0,GUIA LOPES DA LAGUNA,MS,9912.0,Ocupante não especificado de um veículo especi...
2994,2995,2,2026-01-01,1988-10-01,437,1,NaN,1,2,716525,150420,150420,V299,1.0,MARABÁ,PA,271321.0,Motociclista [qualquer] traumatizado em um aci...
3122,3123,2,2026-01-14,1971-12-24,454,1,4,5,0,773120,150050,150050,W209,1.0,ALMEIRIM,PA,38843.0,"Impacto causado por objeto lançado, projetado ..."
3137,3138,2,2026-01-13,2005-08-11,420,1,4,1,4,631105,150600,150600,W136,1.0,PRAINHA,PA,35655.0,Queda de ou para fora de edifícios ou outras e...


**10 maiores causas de morte por acidentes de trabalho em 2026:**

In [108]:
acidentes['DESCRICAO_CID'].value_counts().head(10)

DESCRICAO_CID
Motociclista traumatizado em colisão com um automóvel [carro], "pick up" ou caminhonete - condutor traumatizado em um acidente de trânsito                                                   63
Pessoa traumatizada em um acidente de trânsito com um veículo a motor não especificado                                                                                                       55
Queda de ou para fora de edifícios ou outras estruturas - local não especificado                                                                                                             33
Motociclista traumatizado em colisão com um veículo de transporte pesado ou um ônibus - condutor traumatizado em um acidente de trânsito                                                     31
Ocupante de um veículo de transporte pesado traumatizado em um acidente de transporte sem colisão - condutor [motorista] traumatizado em um acidente de trânsito                             30
Exposição a corrente elétr